## Imports and Environment

In [1]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import VectorStoreIndex, Settings, PromptTemplate, StorageContext, SQLDatabase, SimpleDirectoryReader
from llama_index.core.utilities.sql_wrapper import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine, KnowledgeGraphQueryEngine
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step, Context, Event
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.schema import TextNode
from llama_index.core.tools import QueryEngineTool
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.query_engine import NLSQLTableQueryEngine, RetrieverQueryEngine, SQLTableRetrieverQueryEngine
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.objects import SQLTableNodeMapping, ObjectIndex, SQLTableSchema
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_parse import LlamaParse

from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, util

import pandas as pd, re, ast, textwrap
from sqlalchemy import create_engine, text, inspect

from datasets import Dataset

import ragas
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    LLMSQLEquivalence
)
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas import evaluate, EvaluationDataset
from ragas.metrics import DataCompyScore

from dotenv import load_dotenv, find_dotenv
from typing import Dict, Any, Tuple, Optional, List
import torch
import os
import re
import json
import csv
from tqdm import tqdm

# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# Change the current working directory to the project root
os.chdir(project_root_path)


# --- FIX 2: Bypass find_dotenv() and use a direct, verified path ---
dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# Add a critical check to ensure the .env file exists at this path
if not os.path.exists(dotenv_path):
    raise FileNotFoundError(
        f"CRITICAL ERROR: .env file NOT FOUND at the expected path: {dotenv_path}\n"
        f"Please double-check the path you pasted into 'project_root_path'."
    )


# Load the .env file from the explicit, verified path
load_dotenv(dotenv_path=dotenv_path)

# The project root is now simply the current working directory
project_root = os.getcwd()

# --- Now, the rest of your variable loading will work correctly ---
relative_data_dir = os.getenv("SQL_DATASET_DIR")

# Add a check to make sure the variable was loaded successfully from the file
if not relative_data_dir:
    raise ValueError(
        "ERROR: 'SQL_DATASET_DIR' was not found in your .env file, or the file is empty."
    )

data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# --- Final Verification ---
print(f"✅ Project root successfully set to: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

/anaconda/envs/py311-docext/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Project root successfully set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone
✅ .env file loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env
📁 Data directory set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/SQL_Dataset


## Helper Function to Sanitize File Names

In [2]:
def sanitize_table_name(filename):
    """
    Cleans a filename to create a safe, SQL-compliant table name.
    - Converts to lowercase
    - Replaces spaces and hyphens with underscores
    - Removes all other non-alphanumeric characters (except underscores)
    """
    # Remove the .csv extension
    name = os.path.splitext(filename)[0]
    # Convert to lowercase and replace spaces/hyphens
    name = name.lower().replace(' ', '_').replace('-', '_')
    # Remove any remaining invalid characters
    name = re.sub(r'[^a-z0-9_]', '', name)
    return name

In [3]:
# Create an in-memory SQLite database
# This database exists only as long as the script is running
engine = create_engine("sqlite:///:memory:")

# --- Dynamically load all CLEANED CSVs from the 'SQL_Dataset' directory ---
# This should point to the folder where your 'run_SQL_cleaning.py' script saved the files.
sql_data_directory = "SQL_Dataset" 
table_names = [] # To keep track of the tables we create

print(f"Searching for cleaned CSV files to ingest in '{sql_data_directory}'...")

# Check if the directory exists to avoid errors
if not os.path.isdir(sql_data_directory):
    print(f"Error: The directory '{sql_data_directory}' was not found. Please ensure the cleaning script ran successfully.")
else:
    for filename in os.listdir(sql_data_directory):
        if filename.endswith(".csv"):
            try:
                file_path = os.path.join(sql_data_directory, filename)
                
                # 1. Load the already-cleaned CSV into a DataFrame
                cleaned_df = pd.read_csv(file_path)
                
                # 2. Create a clean table name from the filename
                # Example: "cleaned_m891481.csv" -> "cleaned_m891481"
                table_name = sanitize_table_name(filename)
                table_names.append(table_name)
                
                # 3. Ingest the cleaned DataFrame into the SQL database
                cleaned_df.to_sql(table_name, engine, index=False, if_exists='replace')
                
                print(f" - Successfully ingested '{filename}' into SQL table '{table_name}'")
            except Exception as e:
                print(f" - FAILED to ingest {filename}. Error: {e}")

print(f"\nIn-memory SQL database created and populated with {len(table_names)} table(s).")
sql_database = SQLDatabase(engine)


Searching for cleaned CSV files to ingest in 'SQL_Dataset'...
 - Successfully ingested 'Convicted Penal Population by Age Group (2006-2020).csv' into SQL table 'convicted_penal_population_by_age_group_2006_2020'
 - Successfully ingested 'Convicted Penal Population by Age Group (2020 onwards).csv' into SQL table 'convicted_penal_population_by_age_group_2020_onwards'
 - Successfully ingested 'Convicted Penal Population by Age Group and Offence Group (2006-2020).csv' into SQL table 'convicted_penal_population_by_age_group_and_offence_group_2006_2020'
 - Successfully ingested 'Convicted Penal Population by Age Group and Offence Group (2020 onwards).csv' into SQL table 'convicted_penal_population_by_age_group_and_offence_group_2020_onwards'
 - Successfully ingested 'Convicted Penal Population by Education Level.csv' into SQL table 'convicted_penal_population_by_education_level'
 - Successfully ingested 'Convicted Penal Population by Gender and Offence Group.csv' into SQL table 'convicted_pe

## Llama 3.1 8B Instruct

In [4]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Initialize the tokenizer to get the token ID for stop sequence
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
# The semicolon is the stop character, get its token ID
semicolon_token_id = tokenizer.convert_tokens_to_ids(";")

# Initialize the LLM with the correct stop condition
llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    model_kwargs={"token": hf_token, "dtype": torch.bfloat16},
    # Use 'eos_token_id' which is the correct parameter for this purpose
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # Stop generating as soon as it outputs a semicolon
        "eos_token_id": semicolon_token_id,
    }
)

print("HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.")

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.73it/s]


HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.


## Query Time Table Retrieval

In [5]:
inspector = inspect(engine)

# Create SQLTableNodeMapping and ObjectIndex for table retrieval
table_node_mapping = SQLTableNodeMapping(sql_database)
all_table_schema_objs = [
    SQLTableSchema(table_name=name) for name in inspector.get_table_names()
]

# Create a vector index over the table schemas for retrieval
obj_index = ObjectIndex.from_objects(
    all_table_schema_objs,
    table_node_mapping,
    index_cls=VectorStoreIndex,
)
obj_retriever = obj_index.as_retriever(similarity_top_k=3)

# Use the object retriever to dynamically find the
# right tables to use based on the user's query
query_engine_table_retrieval = SQLTableRetrieverQueryEngine(
    sql_database, obj_retriever
)

# --- Example Usage for Table Retrieval ---
print("\n--- Testing Text-to-SQL with Query-Time Table Retrieval ---")
query_1 = "In 2010, what was the number of male inmates?"
response_1 = query_engine_table_retrieval.query(query_1)
print(f"Query: {query_1}")
print(f"Response: {response_1}\n")
print(f"Generated SQL: {response_1.metadata['sql_query']}\n")


--- Testing Text-to-SQL with Query-Time Table Retrieval ---
Query: In 2010, what was the number of male inmates?
Response: In 2010, there were a total of 10,156 male inmates.

Generated SQL: SELECT number_of_population
FROM convicted_penal_population_by_gender
WHERE year = 2010 AND population_by_gender = 'Male'
ORDER BY number_of_population DESC;



## Query Time Row Retrieval

In [6]:
# -Create Vector Indices for Rows in Each Table
table_names = inspector.get_table_names()
table_row_indices: Dict[str, VectorStoreIndex] = {}
table_row_query_engines: Dict[str, RetrieverQueryEngine] = {}

print("\n--- Starting Row-Level Indexing for Each Table ---")
for table_name in table_names:
    print(f"Processing table: {table_name}")
    # Read table into pandas DataFrame
    df = pd.read_sql_table(table_name, engine)

    # Create TextNode objects for each row
    row_nodes: List[TextNode] = []
    for i, row in df.iterrows():
        # Combine all columns of the row into a single text string
        row_text = " | ".join(map(str, row.values))
        node = TextNode(
            text=f"Row for table '{table_name}': {row_text}",
            metadata={"table_name": table_name, "row_index": i},
        )
        row_nodes.append(node)

    # Create a VectorStoreIndex from the row nodes
    row_index = VectorStoreIndex(row_nodes)
    table_row_indices[table_name] = row_index

    # Create a query engine for this table's rows
    table_row_query_engines[table_name] = row_index.as_query_engine(
        similarity_top_k=5
    )
print("--- Row-Level Indexing Complete ---\n")

# Create QueryEngineTools for Each Table's Row Data
query_engine_tools: List[QueryEngineTool] = []
for table_name, query_engine in table_row_query_engines.items():
    tool_metadata = f"This tool provides access to the rows of the '{table_name}' table. Use it to find specific values or examples within the table."
    tool = QueryEngineTool.from_defaults(
        query_engine=query_engine,
        name=f"row_retriever_{table_name}",
        description=tool_metadata,
    )
    query_engine_tools.append(tool)

# This engine combines both table retrieval and row retrieval.
# obj_retriever` finds the right tables
# tools (row retrievers) help find the right values within those tables

final_query_engine = SQLTableRetrieverQueryEngine(
    sql_database,
    obj_retriever,
    tools=query_engine_tools, # Tools for row-level retrieval
)

# --- Example Usage for Row Retrieval ---
print("--- Testing Text-to-SQL with Query-Time Row-Level Retrieval ---")
query_2 = "In the table for education level, what was the population for those with 'No Formal Education / Lower Primary' in 2021?"
response_2 = final_query_engine.query(query_2)
print(f"Query: {query_2}")
print(f"Response: {response_2}\n")
print(f"Generated SQL: {response_2.metadata['sql_query']}")


--- Starting Row-Level Indexing for Each Table ---
Processing table: convicted_penal_population_by_age_group_2006_2020
Processing table: convicted_penal_population_by_age_group_2020_onwards
Processing table: convicted_penal_population_by_age_group_and_offence_group_2006_2020
Processing table: convicted_penal_population_by_age_group_and_offence_group_2020_onwards
Processing table: convicted_penal_population_by_education_level
Processing table: convicted_penal_population_by_gender
Processing table: convicted_penal_population_by_gender_and_offence_group
Processing table: convicted_penal_population_by_offence_group
--- Row-Level Indexing Complete ---

--- Testing Text-to-SQL with Query-Time Row-Level Retrieval ---
Query: In the table for education level, what was the population for those with 'No Formal Education / Lower Primary' in 2021?
Response: The query did not return any results for the population with 'No Formal Education / Lower Primary' in 2021.

Generated SQL: SELECT number_of_p

## Benchmarking

In [ ]:
# # --- Configuration ---
# relative_benchmark_path = os.getenv("SQL_BENCHMARK_DATASET_DIR")
# if not relative_benchmark_path:
#     raise ValueError("SQL_BENCHMARK_DATASET_DIR not set in .env")

# BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
# OUTPUT_FILENAME = "(3)adv_sql_benchmark_results.csv"
# OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# # --- Load Data ---
# print(f"Loading benchmark data from {BENCHMARK_FILE_PATH}...")
# try:
#     benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
# except UnicodeDecodeError:
#     benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding="latin1")

# # Uncomment to run a smaller test
# # benchmark_df = benchmark_df.head(3)

# # --- Prepare DataFrame for Ragas ---
# # Rename gt_answer to ground_truth for Ragas answer metrics
# if 'gt_answer' in benchmark_df.columns:
#     benchmark_df['gt_answer'] = benchmark_df['gt_answer'].fillna('')
#     benchmark_df = benchmark_df.rename(columns={"gt_answer": "ground_truth"})

# # Prepare ground_truths for RAG metrics (though not used by DataCompy)
# # and gt_query for DataCompy reference execution
# if 'gt_query' in benchmark_df.columns:
#     benchmark_df['gt_query'] = benchmark_df['gt_query'].fillna('')
#     benchmark_df = benchmark_df.rename(columns={"gt_query": "ground_truths"})
#     benchmark_df['ground_truths_list'] = benchmark_df['ground_truths'].apply(lambda x: [x] if isinstance(x, str) else [])
# else:
#     raise ValueError("'gt_query' column not found in the benchmark file.")

# print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")

# # --- Generate Predictions and Collect Data for All Metrics ---
# print("--- Running pipeline and collecting data for evaluation ---")
# results_data = []
# for index, row in tqdm(benchmark_df.iterrows(), total=benchmark_df.shape[0]):
#     question = row['question']
#     ground_truth_sql = row['ground_truths'] # This is the raw SQL string

#     final_response = final_query_engine.query(question)
#     generated_sql = final_response.metadata.get('sql_query', 'No SQL Query Generated')
#     generated_answer = str(final_response)
#     contexts = [node.get_content() for node in final_response.source_nodes]

#     # --- Data for DataCompyScore: Execute both SQL queries ---
#     predicted_csv = ""
#     reference_csv = ""
#     sql_error_log = "OK"

#     try:
#         if generated_sql != 'No SQL Query Generated':
#             predicted_df = pd.read_sql_query(generated_sql, engine)
#             if predicted_df.empty:
#                 sql_error_log = "Generated SQL returned an empty result."
#             else:
#                 predicted_csv = predicted_df.to_csv(index=False)
#         else:
#             sql_error_log = "Pipeline did not generate SQL."

#         if ground_truth_sql:
#             reference_df = pd.read_sql_query(ground_truth_sql, engine)
#             if reference_df.empty:
#                 sql_error_log = "Ground truth SQL returned an empty result."
#             else:
#                 reference_csv = reference_df.to_csv(index=False)
#         else:
#             sql_error_log = "Ground truth SQL is missing."
            
#     except Exception as e:
#         print(f"!!! SQL EXECUTION FAILED for question: '{question}'")
#         print(f"!!! DATABASE ERROR: {e}")
#         print("-" * 20)
#         sql_error_log = str(e)

#     results_data.append({
#         "question": question,
#         "answer": generated_answer,
#         "contexts": contexts,
#         "ground_truth": row.get('ground_truth'),
#         "ground_truths": row.get('ground_truths_list'),
#         "gt_query_str": ground_truth_sql,
#         "generated_sql": generated_sql,
#         "predicted_csv_output": predicted_csv,
#         "reference_csv_output": reference_csv,
#         "sql_execution_error": sql_error_log  # Add the error to the results
#     })

# results_df = pd.DataFrame(results_data)

# # --- CRITICAL DEBUGGING STEP ---
# print("\n--- Inspecting Data Sent to Ragas ---")
# print("This table shows the CSV data that DataCompyScore will receive.")
# print("If these columns are empty or show errors, the score will be NaN.")
# print(results_df[[
#     "predicted_csv_output", 
#     "reference_csv_output", 
#     "sql_execution_error"
# ]].head())
# print("-------------------------------------\n")


# ragas_dataset = Dataset.from_pandas(results_df)

# # --- Configure Ragas Metrics ---
# rag_metrics = [
#     answer_relevancy,
#     faithfulness,
#     context_precision,
#     context_recall,
# ]

# datacompy_metric = DataCompyScore()

# # --- Run Evaluations in Two Parts ---
# print("Evaluating RAG metrics (answer_relevancy, faithfulness, etc.)...")
# rag_result = evaluate(
#     dataset=ragas_dataset,
#     metrics=rag_metrics,
# )
# rag_scores_df = rag_result.to_pandas()
# print("RAG evaluation complete.")

# print("Evaluating SQL data equivalence with DataCompyScore...")
# datacompy_result = evaluate(
#     dataset=ragas_dataset, 
#     metrics=[datacompy_metric],
#     column_map={
#         "response": "predicted_csv_output",
#         "reference": "reference_csv_output",
#     }
# )
# datacompy_scores_df = datacompy_result.to_pandas()
# print("DataCompyScore evaluation complete.")

# # --- Format and Save Final Results ---
# # Merge scores from both evaluations into the results dataframe
# final_df = results_df.copy()
# rag_metric_names = [m.name for m in rag_metrics]
# final_df = final_df.join(rag_scores_df[rag_metric_names])

# datacompy_score_column_name = datacompy_scores_df.columns[-1]
# final_df = final_df.join(datacompy_scores_df[[datacompy_score_column_name]])

# # --- Print Overall Performance Metrics First ---
# print("\n--- Overall Performance Metrics ---")
# all_metric_names = rag_metric_names + [datacompy_score_column_name]
# print(final_df[all_metric_names].mean(numeric_only=True))
# print("----------------------------------")

# # --- Prepare the DataFrame for Final CSV Output ---
# columns_to_keep = [
#     "question",
#     "ground_truth",
#     "gt_query_str",
#     "answer",
#     "generated_sql",
#     "answer_relevancy",
#     "faithfulness",
#     "context_recall",
#     "context_precision",
#     datacompy_score_column_name  # Use the dynamically retrieved column name
# ]
# final_df = final_df[columns_to_keep].copy()

# # Rename columns for final clarity
# final_df.rename(columns={
#     "ground_truth": "gt_answer",
#     "gt_query_str": "gt_query",
#     "answer": "generated_answer"
# }, inplace=True)

# # Save the final, clean dataframe
# final_df.to_csv(OUTPUT_FILE_PATH, index=False)
# print(f"\nClean benchmark results saved to {OUTPUT_FILE_PATH}")

Loading benchmark data from /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Benchmark Dataset/sql_benchmark.csv...
Loaded 100 question-answer pairs for evaluation.
--- Running pipeline and collecting data for evaluation ---


 14%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          | 14/100 [00:36<03:20,  2.33s/it]

!!! SQL EXECUTION FAILED for question: 'How many female inmates were convicted for 'Commercial Crimes' in 2020?'
!!! DATABASE ERROR: (sqlite3.OperationalError) no such column: population_by_crime
[SQL: SELECT number_of_population
FROM convicted_penal_population_by_gender
WHERE year = 2020 AND population_by_gender = 'Female' AND population_by_crime = 'Commercial Crimes'
ORDER BY number_of_population DESC;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
--------------------


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [04:06<00:00,  2.46s/it]



--- Inspecting Data Sent to Ragas ---
This table shows the CSV data that DataCompyScore will receive.
If these columns are empty or show errors, the score will be NaN.
                           predicted_csv_output  \
0                 number_of_population\n10156\n   
1                   number_of_population\n464\n   
2                  number_of_population\n4740\n   
3                    number_of_population\n64\n   
4  population_2020,population_2021\n3426,2938\n   

                                reference_csv_output sql_execution_error  
0                         number_of_population\n39\n                  OK  
1                        number_of_population\n612\n                  OK  
2                       number_of_population\n4740\n                  OK  
3                         number_of_population\n64\n                  OK  
4  year,number_of_population\n2020,3426\n2021,2938\n                  OK  
-------------------------------------

Evaluating RAG metrics (answer_rele

Evaluating:   1%|███████▏                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              | 3/400 [00:01<02:37,  2.51it/s]E

RAG evaluation complete.
Evaluating SQL data equivalence with DataCompyScore...


Evaluating:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              | 0/100 [00:00<?, ?it/s]/

DataCompyScore evaluation complete.

--- Overall Performance Metrics ---
answer_relevancy                 0.952915
faithfulness                     0.072500
context_precision                0.770000
context_recall                   0.740000
data_compare_score(mode=rows)    0.977595
dtype: float64
----------------------------------

Clean benchmark results saved to /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/(3)adv_sql_benchmark_results.csv


In [8]:
# --- 1. Configuration for Benchmark Data ---
# Ensure the SQL_BENCHMARK_DATASET_DIR is set in your .env file
relative_benchmark_path = os.getenv("SQL_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("SQL_BENCHMARK_DATASET_DIR not set in .env file")

BENCHMARK_FILE_PATH = os.path.join(os.getcwd(), relative_benchmark_path)

# --- 2. Load and Prepare the Benchmark DataFrame ---
print(f"Loading benchmark data from {BENCHMARK_FILE_PATH}...")
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding="latin1")

# Rename columns to match Ragas expectations
if 'gt_answer' in benchmark_df.columns:
    benchmark_df['gt_answer'] = benchmark_df['gt_answer'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_answer": "ground_truth"})

if 'gt_query' in benchmark_df.columns:
    benchmark_df['gt_query'] = benchmark_df['gt_query'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_query": "ground_truths"})
    # Ragas expects 'ground_truths' to be a list of strings
    benchmark_df['ground_truths_list'] = benchmark_df['ground_truths'].apply(lambda x: [x] if isinstance(x, str) else [])
else:
    raise ValueError("'gt_query' column not found in the benchmark file.")

print(f"✅ Loaded {len(benchmark_df)} question-answer pairs for evaluation.")


# --- 3. Configure Ragas Metrics ---
# These metrics evaluate the quality of the generated text answers
rag_metrics = [
    answer_relevancy,
    faithfulness,
    context_precision,
    context_recall,
]

# This metric compares the data output of the generated SQL vs. the ground truth SQL
datacompy_metric = DataCompyScore()

print("✅ Benchmark DataFrame and Ragas metrics are configured and ready.")

Loading benchmark data from /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Benchmark Dataset/sql_benchmark.csv...
✅ Loaded 100 question-answer pairs for evaluation.
✅ Benchmark DataFrame and Ragas metrics are configured and ready.


## Benchmarking Table Retrieval

In [18]:
# Inspect the database to get table names
inspector = inspect(engine)
table_names = inspector.get_table_names()

# Create SQLTableSchema objects for each table
all_table_schema_objs = [SQLTableSchema(table_name=name) for name in table_names]

# Create the node mapping and a searchable object index over the table schemas
table_node_mapping = SQLTableNodeMapping(sql_database)
obj_index = ObjectIndex.from_objects(
    all_table_schema_objs,
    table_node_mapping,
    index_cls=VectorStoreIndex,
)

# Create an object retriever that will find the best table(s) for a query
obj_retriever = obj_index.as_retriever(similarity_top_k=3)

# This is the dedicated query engine for table retrieval
# It has no access to row-level tools or data.
query_engine_table_retrieval_only = SQLTableRetrieverQueryEngine(
    sql_database,
    obj_retriever,
    llm=llm # Assumes 'llm' is defined in your notebook
)

print("✅ Engine created. It will first retrieve relevant tables, then generate SQL.")


# --- 2. Run the Benchmark ---
TABLE_ONLY_OUTPUT_FILENAME = "(3 table)adv_sql_benchmark_results.csv"
TABLE_ONLY_OUTPUT_FILE_PATH = os.path.join(os.getcwd(), TABLE_ONLY_OUTPUT_FILENAME)

print(f"\n--- Running Benchmark for Table-Retrieval-Only ---")
print(f"Results will be saved to {TABLE_ONLY_OUTPUT_FILE_PATH}")

table_retrieval_results_data = []
# Assumes 'benchmark_df' is loaded and prepped from the setup cell
for index, row in tqdm(benchmark_df.iterrows(), total=benchmark_df.shape[0], desc="Table-Retrieval-Only Benchmark"):
    question = row['question']
    ground_truth_sql = row['ground_truths']

    # *** Use the new, correct query engine ***
    final_response = query_engine_table_retrieval_only.query(question)

    generated_sql = final_response.metadata.get('sql_query', 'No SQL Query Generated')
    generated_answer = str(final_response)
    contexts = [node.get_content() for node in final_response.source_nodes]
    
    predicted_csv, reference_csv, sql_error_log = "", "", "OK"
    try:
        if generated_sql != 'No SQL Query Generated':
            predicted_df = pd.read_sql_query(generated_sql, engine)
            predicted_csv = predicted_df.to_csv(index=False)
        else:
            sql_error_log = "Pipeline did not generate SQL."
        
        if ground_truth_sql:
            reference_df = pd.read_sql_query(ground_truth_sql, engine)
            reference_csv = reference_df.to_csv(index=False)
            
    except Exception as e:
        sql_error_log = str(e)

    table_retrieval_results_data.append({
        "question": question, "answer": generated_answer, "contexts": contexts,
        "ground_truth": row.get('ground_truth'), "ground_truths": row.get('ground_truths_list'),
        "gt_query_str": ground_truth_sql, "generated_sql": generated_sql,
        "predicted_csv_output": predicted_csv, "reference_csv_output": reference_csv,
        "sql_execution_error": sql_error_log
    })

# --- 3. Evaluate and Save Results ---
results_df_table = pd.DataFrame(table_retrieval_results_data)
ragas_dataset_table = Dataset.from_pandas(results_df_table)

print("\nEvaluating Ragas and DataCompy metrics...")
# Assumes 'rag_metrics' and 'datacompy_metric' are defined
rag_result_table = evaluate(dataset=ragas_dataset_table, metrics=rag_metrics)
datacompy_result_table = evaluate(
    dataset=ragas_dataset_table, metrics=[datacompy_metric],
    column_map={"response": "predicted_csv_output", "reference": "reference_csv_output"}
)

final_df_table = results_df_table.join(rag_result_table.to_pandas()[[m.name for m in rag_metrics]])
datacompy_col_name = datacompy_result_table.to_pandas().columns[-1]
final_df_table = final_df_table.join(datacompy_result_table.to_pandas()[[datacompy_col_name]])

print("\n--- Overall Performance (Table-Retrieval-Only) ---")
all_metric_names = [m.name for m in rag_metrics] + [datacompy_col_name]
print(final_df_table[all_metric_names].mean(numeric_only=True))
print("---------------------------------------------")

final_df_table.to_csv(TABLE_ONLY_OUTPUT_FILE_PATH, index=False)
print(f"✅ Table-retrieval-only benchmark results saved to {TABLE_ONLY_OUTPUT_FILE_PATH}")

✅ Engine created. It will first retrieve relevant tables, then generate SQL.

--- Running Benchmark for Table-Retrieval-Only ---
Results will be saved to /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/(3 table)adv_sql_benchmark_results.csv


Table-Retrieval-Only Benchmark:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          | 0/100 [00:00<?, ?it/s]S


Evaluating Ragas and DataCompy metrics...


Evaluating:   0%|████▊                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 | 2/400 [00:01<03:48,  1.74it/s]E


--- Overall Performance (Table-Retrieval-Only) ---
answer_relevancy                 0.585654
faithfulness                     0.120953
context_precision                0.160000
context_recall                   0.150000
data_compare_score(mode=rows)    0.901090
dtype: float64
---------------------------------------------
✅ Table-retrieval-only benchmark results saved to /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/(3 table)adv_sql_benchmark_results.csv


## Benchmarking Row Retrieval

In [17]:
# --- Create Engine for Row-Aware Retrieval ---
print("Creating a Text-to-SQL engine that uses table schemas AND sample rows for context...")
query_engine_row_aware = NLSQLTableQueryEngine(
    sql_database=sql_database,
    include_sample_rows_in_table_info=True,  # Default behavior, but explicit here
    llm=llm, # Assumes 'llm' is defined in your notebook
)
print("✅ Engine created.")

# --- Benchmarking Configuration ---
ROW_AWARE_OUTPUT_FILENAME = "(3_row)adv_sql_benchmark_results.csv"
ROW_AWARE_OUTPUT_FILE_PATH = os.path.join(os.getcwd(), ROW_AWARE_OUTPUT_FILENAME)

print(f"\n--- Running Benchmark for Row-Aware Retrieval ---")
print(f"Results will be saved to {ROW_AWARE_OUTPUT_FILE_PATH}")

# --- Generate Predictions ---
row_aware_results_data = []
# Assumes 'benchmark_df' is loaded and prepped from the setup cell
for index, row in tqdm(benchmark_df.iterrows(), total=benchmark_df.shape[0], desc="Row-Aware Benchmark"):
    question = row['question']
    ground_truth_sql = row['ground_truths']

    # *** Use the row-aware query engine ***
    final_response = query_engine_row_aware.query(question)

    generated_sql = final_response.metadata.get('sql_query', 'No SQL Query Generated')
    generated_answer = str(final_response)
    contexts = [node.get_content() for node in final_response.source_nodes]
    
    predicted_csv, reference_csv, sql_error_log = "", "", "OK"
    try:
        if generated_sql != 'No SQL Query Generated':
            predicted_df = pd.read_sql_query(generated_sql, engine)
            predicted_csv = predicted_df.to_csv(index=False)
        else:
            sql_error_log = "Pipeline did not generate SQL."
        
        if ground_truth_sql:
            reference_df = pd.read_sql_query(ground_truth_sql, engine)
            reference_csv = reference_df.to_csv(index=False)
            
    except Exception as e:
        sql_error_log = str(e)

    row_aware_results_data.append({
        "question": question, "answer": generated_answer, "contexts": contexts,
        "ground_truth": row.get('ground_truth'), "ground_truths": row.get('ground_truths_list'),
        "gt_query_str": ground_truth_sql, "generated_sql": generated_sql,
        "predicted_csv_output": predicted_csv, "reference_csv_output": reference_csv,
        "sql_execution_error": sql_error_log
    })

# --- Evaluate and Save Results ---
results_df_row = pd.DataFrame(row_aware_results_data)
ragas_dataset_row = Dataset.from_pandas(results_df_row)

print("\nEvaluating Ragas and DataCompy metrics...")
# Assumes 'rag_metrics' and 'datacompy_metric' are defined in the setup cell
rag_result_row = evaluate(dataset=ragas_dataset_row, metrics=rag_metrics)
datacompy_result_row = evaluate(
    dataset=ragas_dataset_row, metrics=[datacompy_metric],
    column_map={"response": "predicted_csv_output", "reference": "reference_csv_output"}
)

final_df_row = results_df_row.join(rag_result_row.to_pandas()[[m.name for m in rag_metrics]])
datacompy_col_name = datacompy_result_row.to_pandas().columns[-1]
final_df_row = final_df_row.join(datacompy_result_row.to_pandas()[[datacompy_col_name]])

print("\n--- Overall Performance (Row-Aware) ---")
all_metric_names = [m.name for m in rag_metrics] + [datacompy_col_name]
print(final_df_row[all_metric_names].mean(numeric_only=True))
print("---------------------------------------")

final_df_row.to_csv(ROW_AWARE_OUTPUT_FILE_PATH, index=False)
print(f"✅ Row-aware benchmark results saved to {ROW_AWARE_OUTPUT_FILE_PATH}")

Creating a Text-to-SQL engine that uses table schemas AND sample rows for context...
✅ Engine created.

--- Running Benchmark for Row-Aware Retrieval ---
Results will be saved to /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/(3_row)adv_sql_benchmark_results.csv


Row-Aware Benchmark:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     | 0/100 [00:00<?, ?it/s]S


Evaluating Ragas and DataCompy metrics...


Evaluating:   0%|██▍                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   | 1/400 [00:01<10:09,  1.53s/it]E


--- Overall Performance (Row-Aware) ---
answer_relevancy                 0.593632
faithfulness                     0.093229
context_precision                0.170000
context_recall                   0.180000
data_compare_score(mode=rows)    0.938413
dtype: float64
---------------------------------------
✅ Row-aware benchmark results saved to /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/(3_row)adv_sql_benchmark_results.csv
